# CropCop Track A — Account Readiness / Evidence Execution

Run with CROPCOP_ACCOUNT_PHASE=readiness first on K1/K2/K3. After the global readiness gate is sealed, rerun with CROPCOP_ACCOUNT_PHASE=evidence. The evidence phase uses two isolated T4 workers and never uses DDP/DataParallel/FSDP.


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

repo = Path(os.environ["CROPCOP_REPO_ROOT"]).resolve()
analysis_sha = os.environ["CROPCOP_ANALYSIS_SHA"].strip()
account_id = os.environ.get("CROPCOP_ACCOUNT_ID", "").strip()
if len(analysis_sha) != 40:
    raise RuntimeError("CROPCOP_ANALYSIS_SHA must be a full 40-character SHA")
observed = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
if observed != analysis_sha:
    raise RuntimeError(f"exact analysis checkout mismatch: expected={analysis_sha}, observed={observed}")
os.environ.setdefault("CROPCOP_NOTEBOOK_STARTED_MONOTONIC", repr(time.monotonic()))
os.environ.setdefault("CROPCOP_NOTEBOOK_HARD_LIMIT_SECONDS", "43200")
os.environ.setdefault("CROPCOP_NOTEBOOK_FINALIZATION_MARGIN_SECONDS", "3600")
print({"analysis_sha": analysis_sha, "account_id": account_id or None, "repo": str(repo)})


In [ ]:
required = [
    "CROPCOP_ACCOUNT_ID", "CROPCOP_ACCOUNT_PHASE", "CROPCOP_MATERIALIZATION_CATALOG",
    "CROPCOP_PLACEMENT_FREEZE", "CROPCOP_ACCOUNT_CONTROL_DIR"
]
missing = [name for name in required if not os.environ.get(name, "").strip()]
if missing:
    raise RuntimeError(f"missing required environment variables: {missing}")
phase = os.environ["CROPCOP_ACCOUNT_PHASE"].strip().lower()
if phase not in {"readiness", "evidence"}:
    raise RuntimeError("CROPCOP_ACCOUNT_PHASE must be readiness or evidence")
control = Path(os.environ["CROPCOP_ACCOUNT_CONTROL_DIR"]).resolve()
control.mkdir(parents=True, exist_ok=True)
inventory = control / f"{account_id}_POSTTRAINING_INVENTORY.json"
readiness = control / f"{account_id}_POSTTRAINING_ACCOUNT_READINESS.json"
subprocess.run([
    sys.executable,
    str(repo / "journal_extension/scripts/build_tracka_v12_posttraining_account_inventory.py"),
    "--repo-root", str(repo),
    "--account-id", account_id,
    "--materialization-catalog", os.environ["CROPCOP_MATERIALIZATION_CATALOG"],
    "--placement-freeze", os.environ["CROPCOP_PLACEMENT_FREEZE"],
    "--analysis-source-git-commit", analysis_sha,
    "--output", str(inventory),
], cwd=repo, check=True)
subprocess.run([
    sys.executable,
    str(repo / "journal_extension/scripts/build_tracka_v12_posttraining_account_readiness.py"),
    "--repo-root", str(repo),
    "--inventory", str(inventory),
    "--analysis-source-git-commit", analysis_sha,
    "--output", str(readiness),
], cwd=repo, check=True)
gate = json.loads(readiness.read_text(encoding="utf-8"))
if gate.get("status") != "PASS":
    raise RuntimeError("account readiness gate is not PASS")
print(json.dumps({"phase": phase, "account_id": account_id, "readiness_gate_sha256": gate["gate_sha256"]}, indent=2))


In [ ]:
if phase == "evidence":
    required = ["CROPCOP_GLOBAL_READINESS", "CROPCOP_ACCOUNT_WORK_ROOT", "CROPCOP_ACCOUNT_COMPLETION"]
    missing = [name for name in required if not os.environ.get(name, "").strip()]
    if missing:
        raise RuntimeError(f"evidence phase missing required environment variables: {missing}")
    global_gate = json.loads(Path(os.environ["CROPCOP_GLOBAL_READINESS"]).read_text(encoding="utf-8"))
    if global_gate.get("status") != "PASS" or global_gate.get("ready_for_posttraining_evidence") is not True:
        raise RuntimeError("global readiness must be PASS before evidence execution")
    cp = subprocess.run([
        sys.executable,
        str(repo / "journal_extension/scripts/run_tracka_v12_posttraining_account.py"),
        "--repo-root", str(repo),
        "--account-id", account_id,
        "--account-inventory", str(inventory),
        "--account-readiness", str(readiness),
        "--global-readiness", os.environ["CROPCOP_GLOBAL_READINESS"],
        "--placement-freeze", os.environ["CROPCOP_PLACEMENT_FREEZE"],
        "--analysis-source-git-commit", analysis_sha,
        "--work-root", os.environ["CROPCOP_ACCOUNT_WORK_ROOT"],
        "--output", os.environ["CROPCOP_ACCOUNT_COMPLETION"],
    ], cwd=repo, check=False)
    if cp.returncode not in {0, 20}:
        raise RuntimeError(f"post-training account operator failed rc={cp.returncode}")
    result = json.loads(Path(os.environ["CROPCOP_ACCOUNT_COMPLETION"]).read_text(encoding="utf-8"))
    print(json.dumps({
        "status": result["status"],
        "account_id": result["account_id"],
        "assigned_state_count": result["assigned_state_count"],
        "completed_state_count": result["completed_state_count"],
        "manifest_sha256": result["manifest_sha256"],
    }, indent=2))
else:
    print("Readiness-only phase complete. Do not start evidence until the global readiness gate is sealed.")
